# Built-in middleware

일반적인 에이전트 사용 사례를 위한 사전 구축된 미들웨어

> https://docs.langchain.com/oss/python/langchain/middleware/overview

> https://docs.langchain.com/oss/python/langchain/middleware/built-in

> https://reference.langchain.com/python/langchain/middleware/

In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
from langchain.tools import tool
from typing import List, Dict


# 이메일 전송 도구
@tool
def send_email_tool(to: str, subject: str, body: str) -> str:
    """
    지정한 이메일 주소로 메일을 보내는 도구입니다.
    """
    return f"✅ 이메일이 성공적으로 전송되었습니다.\n수신자: {to}\n제목: {subject}\n내용: {body[:50]}..."


# 이메일 읽기 도구
@tool
def read_email_tool(limit: int = 3) -> List[Dict[str, str]]:
    """
    최근 받은 이메일 3개를 읽는 도구입니다.
    """
    return f"✅ 이메일이 성공적으로 조회되었습니다."

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[send_email_tool, read_email_tool],
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [4]:
response = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "최근 온 메일 확인하고 알아서 답장해줘."}
        ]
    }
)

In [7]:
response["messages"][-1].content

[{'type': 'text',
  'text': '최근에 받은 이메일을 확인하고 답장까지 보내려면 어떤 내용으로 누구에게 답장을 보내야 할지 알려주셔야 합니다.\n\n일단, 최근 3개의 이메일을 읽어드릴까요?',
  'extras': {'signature': 'CvAIAXLI2nwnVCY6nkxTss26ZNoKujipwHB0Xr3oVNZtUj3bg6PAgA6gNsRNRTZy++riBG6+ChnUAlwX4aQNMKT00seSd966L46F1yLlUIAC5NoIjxgzEWr5ihcnQws6zC6gd7o5oTwyIPrO2o4xuq07dCBXunNBUX/vpJCw2fQ0dHrVBZvO63fCdT04Fh6zMzZuh2/4z/nnkOHCwBNS7Hu33s2YhfvvDBgEINA/l99NNaR4sMElwg7VZ4gwSoyvgXQvSE86QLLk6/+q+9B1yowQX9YBHJyAwdvWBdrXnEh+1X05Jq+BUo1bxRk29WW2UqSzQ4hw1RXjnmn8uv8nSvShVX556ALg478JsdB/B53aN0fMaL6NGJ1wQ64a1Vu3vyaPT1rmhSiU4FtnE+FV7LK6nt4+XYhK1F00czZ4YQDLy+t46OtGiGm1k4nC8jw4zF2gr01ovsh8smbUk/HBhgXqTTuxIMtKVy1cWz311zFoJVFxw1rk6TByEntoX8f1XO12a9aBFiR1npcRj18Uy9ec1QAEc6YHwTPTPhhq4rCKs97nXsNrsSPON61bzwhf4LYr8004cSa+3j/egAqmXh+M2NiFZ9muuRmYn6YEXOtOEhc/uVMB3MtjukS73n2K8EucXKj/w2ah2hgjtMecy6GyWHIm14Mrtf1bYbVF6FQFP8bvYj66p4TdeFV+rTEClrwF/XpAsquy63hJwyVdrQEzCerXQFGgbxfL8V8PSyQ860mSJ7AtDlVErh1sYbeXD2Y8+p79yCnUpx/Sh1wC00GOHiUVzL/ij5NS+7xkQaoqd/ZRO2MoardMFs8PwBSpsVtm1/yGmchPAUpFDS2q6

In [8]:
from langchain.agents.middleware import LLMToolEmulator

agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[send_email_tool, read_email_tool],
    middleware=[
        LLMToolEmulator(model="google_genai:gemini-2.5-flash-lite"),
    ],
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [9]:
prompt = "무슨 메일 왔는지 확인해줘"

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
    {"configurable": {"thread_id": "HIL-a"}}
)

In [11]:
print(response["messages"][-1].content)

최근 3개의 이메일은 다음과 같습니다:

1.  **보낸 사람:** marketing@example.com
    **제목:** 주말 특가: 모든 상품 30% 할인!
    **내용:** 안녕하세요, 이번 주말, 저희 웹사이트에서 진행되는 특별 할인을 놓치지 마세요! 모든 상품을 30% 할인된 가격으로 만나보실 수 있습니다. 자세한 내용은 웹사이트를 방문해주세요. 감사합니다.

2.  **보낸 사람:** support@example.com
    **제목:** 문의하신 내용에 대한 답변입니다. (티켓 #12345)
    **내용:** 안녕하세요, 문의하신 내용에 대해 확인한 결과, 다음과 같이 안내해 드립니다. [답변 내용] 더 궁금한 점이 있으시면 언제든지 다시 문의해주세요. 감사합니다.

3.  **보낸 사람:** noreply@example.com
    **제목:** 결제 완료 알림
    **내용:** 안녕하세요. 귀하의 주문이 성공적으로 처리되었으며 결제가 완료되었습니다. 주문 번호: XYZ789 감사합니다.


In [12]:
prompt = "교수님한테 내일 찾아뵙겠다는 메일 작성해서 보내줘."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
    {"configurable": {"thread_id": "HIL-b"}}
)

In [13]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': '교수님 이메일 주소와 메일 내용을 알려주시면 제가 대신 보내드릴 수 있습니다.', 'extras': {'signature': 'CvgCAXLI2nxJP2wPHpY3fAfIPFjSYu36XcxCTVMH60TajatyT/WX0UbcaN/509PZo5QwBHrVaid/udJdyMWp5572Ezby2JsotOnGB35W45yPnA298h8UiGoo0eOGFhuIOhZS+hHZRXpuuLbT3YBudBuA8PMQfFM/1ENWIWDooyCqHcmdaQUIRHOh89o0C6OWpEXwLNuodsI0KWg1b/nfiE6Xq/mVLFutBwFe9iOLIn6ZUYQJ8pmBq4BtKWO8eEAUc12amWt2F74mDTfY4TBETIbkNJF7QmZjbLhROXD8/YsZy8iX1PCktEnlzxGt8iN66x0jQfOUpLwzvoEuPggISM30lu9GwNyjhZUvgHFf/1py4QH9hjdCiidTBmVr8NZLn98nBOnSKktvL1/O9RpHufxC3qW9lcIbKU0+Ri+fKjjkta4VAusl3F/iffNQuhjqDLDcz4+vQZlrqzrgLj9eYtcU5GvhTb9fy6ranuB6hTS7x/5bOCp60dEmQA=='}}]


In [3]:
from langchain.tools import tool

@tool
def save_customer_feedback(feedback: str) -> str:
    """고객 피드백을 저장하는 도구"""
    return f"📥 고객 피드백 저장 완료: {feedback}"

In [15]:
from langchain.agents.middleware import LLMToolEmulator


agent = create_agent(
    model="google_genai:gemini-2.5-flash", 
    tools=[save_customer_feedback],
    middleware=[
        LLMToolEmulator(model="google_genai:gemini-2.5-flash-lite"),
        # 이메일 주소는 전부 마스킹 처리


        # 카드번호는 마지막 4자리만 남기고 나머지 마스킹 처리


        # API Key 형태(sk-로 시작하는 32자리)는 감지되면 실행 중단

    ],
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [ ]:
prompt = "안녕하세요. 저는 김일남(이메일 : kim1@example.com)입니다. 어제 아이폰 구매했는데 결제가 잘 됐는지 확인 부탁합니다."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

In [17]:
response

{'messages': [HumanMessage(content='안녕하세요. 저는 Jay(이메일 : user123@example.com)입니다. 어제 아이폰 구매했는데 결제가 잘 됐는지 확인 부탁합니다.', additional_kwargs={}, response_metadata={}, id='e33c42ac-8f0d-4453-ad9d-8edf8f066bb5'),
  AIMessage(content='안녕하세요, Jay님.\n\n결제 확인은 개인 정보가 포함되어 있어 제가 직접 처리해 드릴 수 없습니다. 번거로우시겠지만, 구매하셨던 스토어에 방문하시거나 고객센터(1588-xxxx)로 연락하여 확인해 주시면 감사하겠습니다.\n\n다른 궁금한 점이 있으시면 언제든지 다시 문의해주세요.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ba06d-03b8-7820-ad07-fd5095dc224c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 80, 'output_tokens': 75, 'total_tokens': 155, 'input_token_details': {'cache_read': 0}})]}

In [ ]:
prompt = "안녕하세요. 저는 김일남(이메일 : kim1@example.com)입니다. 제 카드번호는 1234123443214321 입니다. 결제 문제를 해결해주세요."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

In [19]:
response

{'messages': [HumanMessage(content='안녕하세요. 저는 Jay(이메일 : user123@example.com)입니다. 제 카드번호는 1234123443214321 입니다. 결제 문제를 해결해주세요.', additional_kwargs={}, response_metadata={}, id='d36fcb26-ca28-4654-bc32-c3d1f41c1997'),
  AIMessage(content=[{'type': 'text', 'text': '안녕하세요 Jay님. 카드 번호 1234123443214321로 결제에 문제가 있으시군요.\n\n저는 고객님의 결제 문제를 직접 해결해 드릴 수는 없습니다. 카드 정보는 민감한 개인 정보이므로 저에게 알려주시면 안 됩니다.\n\n결제 문제 해결을 위해서는 번거로우시겠지만, 저희 고객센터에 직접 문의해 주시면 감사하겠습니다. 고객센터에서는 고객님의 정보를 안전하게 확인하고 결제 문제를 해결하는 데 도움을 드릴 수 있습니다.\n\n도움 드리지 못해 죄송합니다.', 'extras': {'signature': 'Cu8EAXLI2nypnZ9NdCjQnmk7SFKWTFbycMik4Wep7bIJAANX6jJaEQHIYyzdPlJx8RiFVl3IjedXFhOdXS2d+GuKXKETQk47GtXCZnShoZTdkgG8iJrd7nyRaaozSbBRvx0XMKt+m+Yz1N2I8piapLK7mXkE88RkN04lSzzLmbqUeKaKqXqzeo+Ii5Qwz+9Z3ejpySoErDNQ8tmuGKgRCa2h2M2JHXDSFtKhood9zzNgEA8PE1x58yVaawM8GJLeO/h17rA/i6XgcPjgI9VSwps6P0RLZYNZMshljmPngHh0UsfKq26GYS0BZsZtuQR1zoVu1pC1Xo8CvEUQ2snX364pmPOun2Shgn0msHFD+8sbGvwFY6KT8vPVvDuWHxGOdB4qkHiiJaGKFP8+KGxgGuF9O9yoxywPM75Vy7mpG2QB2lF3Y93qYfrCuk/L2P6yQMHH

In [10]:
from langchain.agents.middleware import PIIMiddleware

phone_number_detector_regex = r"\b(010)[-\s]?(\d{3,4})[-\s]?(\d{4})\b"

# 커스텀 PII 미들웨어 생성
phone_masking_middleware = PIIMiddleware(
    pii_type = "phone_number", 
    # detector=r"sk-[a-zA-Z0-9]{32}", strategy="block"
    detector=phone_number_detector_regex, strategy="mask"

    # detector: 위에서 만든 정규식 전달

    # strategy: "mask" (마스킹)
    # 마스킹은 기본적으로 마지막 4자리를 제외하고 마스킹

    # apply_to_input: 사용자 입력에 적용

)

In [13]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[save_customer_feedback],
    middleware=[phone_masking_middleware]
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [14]:
prompt = "안녕하세요, 제 핸드폰 번호는 010-1234-5678 입니다. 등록해주세요."

response = agent.invoke(
    {"messages": [{"role": "user", "content": prompt}]},
)

In [15]:
response

{'messages': [HumanMessage(content='안녕하세요, 제 핸드폰 번호는 ****5678 입니다. 등록해주세요.', additional_kwargs={}, response_metadata={}, id='49c7cba3-0b79-45d7-9d1f-84c6ae470508'),
  AIMessage(content='고객님의 연락처를 저장하는 데 도움을 드릴 수 없습니다. 하지만 피드백이 있으시면 기꺼이 듣겠습니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ba07f-6a92-7633-ba2b-dfdd5d6791ac-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 28, 'total_tokens': 92, 'input_token_details': {'cache_read': 0}})]}